# CR-2 段2 の固定（CHASE-Math・API なし・CPU）

凍結した計画 `notes/cr2_plan.md`（第10版）の「段2 の固定」を作ります。**API には何も送りません。** 問題文はどこにも表示・保存しません（数・ID・ハッシュだけ）。

- 固定した版の CHASE-Math を読み、**Hugging Face が示す正規のハッシュと、取ってきたファイルのハッシュが一致するか**を確かめます（違えば止まります）。
- README のライセンスの表示（MIT のはず）と sha256 を記録します（G-L：表示による運用の判断。作者への問い合わせはしていません）。
- 下見 3 の全 96 問（CHASE-Math 48 問と MATH 48 問）を作り直して記録のハッシュ `f52bb374…` と照合し、重複を除き、下見の問題を除いて、候補 120 問の順を作ります。

**使い方**：CPU のランタイム。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で 5 分くらい。結果は Drive の `DECISION_POINTS/cr2/STAGE2/CR2_stage2_lock.json`。終わったら Claude に教えてね（Claude がリポジトリに固定します）。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, ancestors, (pinned SDK and key, never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "a9fd5344084cdec8c402f1768cdf993056a7b86f"
ANCESTORS = []
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
for anc in ["7949d74"] + ANCESTORS:
    assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", anc, head]).returncode == 0, anc + " が祖先ではありません"
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from pork import cr2 as C
from pork import cr2_search as S
from pork import cr2_run as RN
from pork import cr2_g2 as G
from pork import cr2_g3 as G3
from pork import cr2_stage3b as T3
from pork import cr1_run as R1
from pork import alt_prelook as P
import numpy
RN.check_prompt()
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr2")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))

def git_blob(path):
    return subprocess.check_output(["git", "hash-object", path], text=True).strip()

def write_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    R1._atomic_write(path, obj)
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_chase_verified():
    """Plan 1. (H4, I3): pinned revision, canonical hash from the Hugging Face API vs the downloaded bytes."""
    from huggingface_hub import HfApi, hf_hub_download
    info = HfApi().dataset_info(C.DATASET_ID, revision=C.DATASET_REVISION, files_metadata=True, token=False)
    sib = next((s for s in info.siblings if s.rfilename == C.DATASET_FILE), None)
    meta = None
    if sib is not None:
        lfs = sib.lfs
        if lfs is not None and not isinstance(lfs, dict):
            lfs = {"sha256": getattr(lfs, "sha256", None), "size": getattr(lfs, "size", None)}
        meta = {"blob_id": sib.blob_id, "lfs": lfs}
    p = hf_hub_download(C.DATASET_ID, C.DATASET_FILE, repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    data = Path(p).read_bytes()
    prov = C.check_provenance(data, meta, info.sha)
    rp = hf_hub_download(C.DATASET_ID, "README.md", repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    readme = Path(rp).read_text(encoding="utf-8")
    rows = C.parse_rows(data)
    del data
    return rows, prov, {"readme_sha256": C.sha256_text(readme), "license": C.readme_license(readme)}

def rebuild_stage2(file_sha=None, cand_sha=None, excl_sha=None):
    rows, prov, readme = load_chase_verified()
    if file_sha is not None:
        assert prov["file_sha256"] == file_sha, "データのファイルが段2 の固定と違います。止めます（1 件も送りません）。"
    pre = C.reproduce_prelook3(rows, P.load_math())
    cand = C.build_candidates(rows, pre)
    if cand_sha is not None:
        assert cand["candidates_sha256"] == cand_sha, "候補の一覧が段2 の固定と違います。止めます。"
    if excl_sha is not None:
        assert cand["excluded_sha256"] == excl_sha, "除外した行が段2 の固定と違います。止めます。"
    by_id = {int(r["ID"]): r for r in rows}
    cands = [{**c, "question": C.problem_text(by_id[c["id"]]), "gold": str(C.gold(by_id[c["id"]]))}
             for c in cand["candidates"]]                                     # PRIVATE, memory only
    del rows, by_id
    return cand, cands, prov, readme, pre

print("study:", C.STUDY, "| exact head:", head, "| numpy", numpy.__version__, "| python", sys.version.split()[0])

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-2 段2 の固定（CHASE-Math・API なし・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-2 stage2 lock', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — 段2: numbers, ids and hashes only.
t0 = time.time()
cand, cands, prov, readme, pre = rebuild_stage2()
del cands
assert readme["license"] is not None and readme["license"].lower() == "mit", "README のライセンスの表示が MIT ではありません。止めます。"
GL_RECORD = {
    "basis": "表示による運用の判断（［決定B'］）。作者への個別の問い合わせはしていない。表示だけで、すべての権利や外部 API に送る条件が完全に確かめられたとは言わない。",
    "hf_card_license_at_revision": readme["license"], "readme_sha256": readme["readme_sha256"],
    "repo_license": "MIT（GitHub McGill-NLP/CHASE の LICENSE の表示。2026-10-13 に Claude が公開ページで確認）",
    "sources": "種の問題は GSM8K・SVAMP の test（どちらもリポジトリの表示は MIT。2026-10-13 に確認）。問題は GPT-4o-mini が生成し、claude-3.5-haiku・gemini-1.5-flash などで確かめたもの（論文）。",
    "citation": "Patel ほか, How to Get Your LLM to Generate Challenging Problems for Evaluation, arXiv:2502.14678",
    "checked_at": dt.datetime.now(JST).isoformat(timespec="seconds")}
lock = {"schema": "cr2-stage2-lock-v2", "study": C.STUDY, "code_head": head, "dataset": C.DATASET_ID,
        "revision": C.DATASET_REVISION, "file": C.DATASET_FILE, "source_url": "https://huggingface.co/datasets/" + C.DATASET_ID,
        "provenance": prov, "file_sha256": prov["file_sha256"], "GL_RECORD": GL_RECORD,
        "rows": cand["rows"], "dedup_dropped": cand["dedup_dropped"], "unreadable_gold_all_rows": cand["unreadable_gold_all_rows"],
        "prelook3_all96_sha256": pre["all96_sha256"], "prelook3_excluded": cand["excluded"],
        "prelook3_excluded_ids_sha256": cand["excluded_sha256"], "unreadable_gold_after_exclusion": cand["unreadable_gold"],
        "pool": cand["pool"], "depth_counts_all_rows": cand["depth_counts_all_rows"],
        "depth_group_counts_candidates": cand["depth_group_counts_candidates"],
        "seeds_all_rows": cand["seeds_all_rows"], "seeds_candidates": cand["seeds_candidates"],
        "seeds_candidates_sharing_prelook_seed": cand["seeds_candidates_sharing_prelook_seed"],
        "candidates": [[c["id"], c["rank"], c["depth"]] for c in cand["candidates"]],
        "candidates_sha256": cand["candidates_sha256"],
        "seeds": {"SEED_CR2_ITEM": C.SEED_CR2_ITEM, "SEED_CR2_WORD": C.SEED_CR2_WORD},
        "budget": {"B_cap": C.B_CAP, "B_stop": C.B_STOP}, "N": {"N_MIN": C.N_MIN, "N_MAX": C.N_MAX},
        "design": {"K": C.K, "K_A": C.K_A, "K_S": C.K_S, "max_tokens": C.MAX_TOKENS},
        "seconds": round(time.time() - t0, 1)}
sha = write_json(D / "STAGE2/CR2_stage2_lock.json", lock)
print("ファイルの証跡:", prov["kind"], "一致 | 行の数:", cand["rows"], "| 重複で落とした:", cand["dedup_dropped"])
print("下見 3 の照合: OK | 除外:", cand["excluded"], "| 候補:", len(cand["candidates"]), "| 深さの層:", cand["depth_group_counts_candidates"])
print("種（候補 120 問）:", {k: cand["seeds_candidates"][k] for k in ("distinct_seeds", "largest_seed", "N_eff")})
print("file_sha256:", prov["file_sha256"])
print("candidates_sha256:", cand["candidates_sha256"])
print("prelook3_excluded_ids_sha256:", cand["excluded_sha256"], "| ロックのファイルの sha256:", sha)
notify('終わりました：CR-2 の段2 の固定を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()